# Goal-Based Warehouse Agent

This notebook implements the coding task in `agents_lab.pdf`. The warehouse is a
deterministic grid, the state is the robot's `(row, column)` position, the available
actions are Up/Down/Left/Right, and the explicit goal is to reach `G` without crossing
a shelf (`#`). Breadth-first search (BFS) is appropriate because every move has equal
cost, so the first solution found is a shortest path.

In [1]:
from collections import deque
from dataclasses import dataclass

WAREHOUSE = [
    "#####################",
    "#S....#............G#",
    "#.##....##########..#",
    "#....##.............#",
    "#.######.###.#.###..#",
    "#........#..........#",
    "#####################",
]

MOVES = {
    "Up": (-1, 0),
    "Down": (1, 0),
    "Left": (0, -1),
    "Right": (0, 1),
}


@dataclass
class SearchResult:
    positions: list[tuple[int, int]]
    actions: list[str]
    states_expanded: int

    @property
    def path_length(self):
        return len(self.actions)


class GoalBasedWarehouseAgent:
    '''A goal-based agent that plans before acting by running BFS.'''

    def __init__(self, grid):
        if not grid or len({len(row) for row in grid}) != 1:
            raise ValueError("The grid must be non-empty and rectangular")
        self.grid = tuple(grid)
        self.rows, self.cols = len(grid), len(grid[0])
        self.start = self._find_unique("S")
        self.goal = self._find_unique("G")

    def _find_unique(self, symbol):
        matches = [
            (r, c)
            for r, row in enumerate(self.grid)
            for c, value in enumerate(row)
            if value == symbol
        ]
        if len(matches) != 1:
            raise ValueError(f"Expected exactly one {symbol!r}, found {len(matches)}")
        return matches[0]

    def successors(self, state):
        for action, (dr, dc) in MOVES.items():
            nxt = (state[0] + dr, state[1] + dc)
            r, c = nxt
            if 0 <= r < self.rows and 0 <= c < self.cols and self.grid[r][c] != "#":
                yield action, nxt

    def find_path(self):
        frontier = deque([self.start])
        parent = {self.start: (None, None)}
        expanded = 0

        while frontier:
            state = frontier.popleft()
            if state == self.goal:
                positions, actions = [], []
                while state is not None:
                    positions.append(state)
                    previous, action = parent[state]
                    if action is not None:
                        actions.append(action)
                    state = previous
                return SearchResult(
                    positions=list(reversed(positions)),
                    actions=list(reversed(actions)),
                    states_expanded=expanded,
                )

            expanded += 1
            for action, nxt in self.successors(state):
                if nxt not in parent:
                    parent[nxt] = (state, action)
                    frontier.append(nxt)
        return None

    def draw_path(self, result):
        canvas = [list(row) for row in self.grid]
        for r, c in result.positions[1:-1]:
            canvas[r][c] = "*"
        return "\n".join("".join(row) for row in canvas)

In [2]:
agent = GoalBasedWarehouseAgent(WAREHOUSE)
result = agent.find_path()

if result is None:
    print("No collision-free path exists.")
else:
    print("Path found:", result.positions)
    print("Actions:", " -> ".join(result.actions))
    print("Path length:", result.path_length)
    print("States expanded:", result.states_expanded)
    print("\nPath drawn on warehouse:\n" + agent.draw_path(result))

Path found: [(1, 1), (1, 2), (1, 3), (1, 4), (2, 4), (2, 5), (2, 6), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (1, 16), (1, 17), (1, 18), (1, 19)]
Actions: Right -> Right -> Right -> Down -> Right -> Right -> Right -> Up -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right
Path length: 20
States expanded: 58

Path drawn on warehouse:
#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


## Independent validation

In [3]:
assert result is not None
assert result.positions[0] == agent.start
assert result.positions[-1] == agent.goal
assert len(result.positions) == result.path_length + 1
for (r1, c1), (r2, c2) in zip(result.positions, result.positions[1:]):
    assert abs(r1 - r2) + abs(c1 - c2) == 1, "Every move must be one grid cell"
    assert WAREHOUSE[r2][c2] != "#", "The path must never enter a shelf"

BLOCKED = [
    "#####",
    "#S#G#",
    "#####",
]
assert GoalBasedWarehouseAgent(BLOCKED).find_path() is None
print("All path-validity checks passed, including the no-solution case.")

All path-validity checks passed, including the no-solution case.
